# Controlling Output & Generating Structured Data (JSON)

When integrating Claude into automated backend pipelines or UI tools (like an AWS EventBridge Rule Generator), you need **pure, raw data** (JSON, code, or lists) and **zero conversational chatter**.

### The Problem:
By default, Claude is naturally conversational and helpful. When asked for JSON, it often outputs:
```markdown
Here is the EventBridge rule you requested:
```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"]
}
```
This rule will notify you when an EC2 instance changes state.
```
Passing this output to `json.loads()` immediately throws `json.decoder.JSONDecodeError`.

### The Anthropic Technique:
Combine **Assistant Message Prefilling** with **Stop Sequences**:
1. **Prefill (`add_assistant_message(messages, "```json")`)**: Claude assumes it has already started the code block and skips all introductory chatter.
2. **Stop Sequence (`stop_sequences=["```"]`)**: As soon as Claude closes the JSON block, generation terminates instantly—cutting off all trailing explanations!

In [ ]:
# Install dependencies
%pip install anthropic python-dotenv

In [ ]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Initialize Anthropic Client
from anthropic import Anthropic

client = Anthropic()
model = "claude-3-5-sonnet-latest"

In [ ]:
# Helper functions with dynamic stop_sequences support
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

--- 
## 1. Stop Sequences in Action
Here we ask Claude to count from 1 to 10, but configure `stop_sequences=["5", "3, 4"]`. As soon as Claude tries to emit any of these tokens, generation stops.

In [ ]:
messages = []
add_user_message(messages, "Count from 1 to 10 separated by commas")
answer = chat(messages, stop_sequences=["5", "3, 4"])
print(repr(answer))

--- 
## 2. The Default Output: Markdown & Conversational Wrapping
Notice how asking for JSON wraps the output in markdown code blocks.

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
default_response = chat(messages)
print(default_response)

--- 
## 3. The Solution: Assistant Prefilling + Stop Sequences
1. `add_assistant_message(messages, "```json")` forces Claude to continue as if it already typed the opening tag.
2. `stop_sequences=["```"]` halts the model the exact moment it completes the JSON block.

In [ ]:
messages = []
add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

clean_text = chat(messages, stop_sequences=["```"])
print("Raw Returned Text:")
print(repr(clean_text))
print("\nFormatted Output:")
print(clean_text)

--- 
## 4. Parsing Cleanly into Python Data Structures
With conversational fluff eliminated, `json.loads()` succeeds seamlessly with zero regex or cleanup.

In [ ]:
import json

parsed_json = json.loads(clean_text.strip())
print("Parsed Python Dictionary:", parsed_json)
print("Type:", type(parsed_json))
print("Event Source:", parsed_json.get("source"))

--- 
## 5. Course Exercise: Structured AWS CLI Commands

### The Challenge:
- Prompt: `"Generate three different sample AWS CLI commands. Each should be very short."`
- Using **only** message prefilling and stop sequences, obtain all three commands in a single response.
- Zero comments, zero conversational explanations, and no modifying the user prompt!

### The Key Insight:
Message prefilling is not limited to backticks (```). You can prefill an assistant statement that commits the model to your exact formatting requirements before generation begins!

In [ ]:
messages = []
prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""
add_user_message(messages, prompt)

# Advanced Assistant Prefill: Commits Claude to single block + zero comments
add_assistant_message(
    messages, 
    "Here are all three commands in a single block without any comments:\n```bash"
)

text = chat(messages, stop_sequences=["```"])
print("--- Final Clean Commands Output ---")
print(text.strip())